# EV motor, inverter and reduction

The CT-17 EV's motor and inverter (`lapsim/subsystems/power/ev.py`): torque limited by the motor, the
inverter's phase current, the accumulator's power limit (80 kW rule or a strategy limit), field weakening
above the corner speed (which moves with pack voltage) and the cells' current limit. Every value here is a
PLACEHOLDER until confirmed in `cars/ct17_ev/vehicle.yaml`.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
CAR = lapsim.load_car("ct17_ev")
SL = [slider(CAR, "ev:motor.peak_torque", 100, 300, 5, "motor peak torque", 1, "N m"),
      slider(CAR, "ev:motor.peak_power", 40, 150, 5, "motor peak power", 1e-3, "kW"),
      slider(CAR, "ev:motor.max_rpm", 4000, 9000, 100, "motor max speed", 1, "rpm"),
      slider(CAR, "ev:drivetrain.rear_sprocket", 30, 55, 1, "rear sprocket", 1, "teeth"),
      slider(CAR, "pack:limits.max_power", 30, 80, 1, "accumulator power limit", 1e-3, "kW"),
      slider(CAR, "ev:regen.max_torque", 0, 150, 5, "regen torque", 1, "N m")]
out, out_lap = W.Output(), W.Output()


def draw(*_):
    with out:
        out.clear_output(wait=True)
        veh = with_values(CAR, SL).build()
        T = veh.tables0
        codes = lapsim.subsystems.power.ev.CODES
        fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
        ax[0].plot(3.6 * T.v, T.F, label="drive"); ax[0].plot(3.6 * T.v, -T.F_regen, label="regen")
        ax[0].plot(3.6 * T.v, T.R, "k--", label="resistance")
        ax[0].set(xlabel="speed [km/h]", ylabel="N", title="Wheel force"); ax[0].legend()
        ax[1].plot(3.6 * T.v, T.P_acc / 1e3); ax[1].set(xlabel="speed [km/h]", ylabel="kW", title="Accumulator power at full drive")
        rg, tg = np.linspace(100, veh.evc.motor.max_rpm, 40), np.linspace(1, veh.evc.motor.peak_torque, 30)
        m = ax[2].contourf(rg, tg, 100 * veh.ep.motor.efficiency_map(rg, tg), levels=np.arange(70, 100, 2))
        plt.colorbar(m, ax=ax[2], label="%"); ax[2].set(xlabel="rpm", ylabel="N m", title="Motor + inverter efficiency")
        plt.show()
        lim = pd.Series([codes[int(c)] for c in T.code]).value_counts()
        print("what limits the drive torque (share of the speed grid):", dict((lim / lim.sum()).round(2)))


def laps():
    display(with_values(CAR, SL).run(("acceleration", "autocross")).table().round(3))


for s in SL:
    s.observe(draw, "value")
draw()
display(W.VBox([*SL, out, run_button("Run acceleration + autocross", laps, out_lap), out_lap]))